# # 01 - Data Cleaning, EDA, Feature Engineering & Aggregation
#
# **Project:** AI-Powered Medical Claim Denial Predictor & Payer Intelligence System
# **Dataset:** DenialIQ - 120K Medical Claims | X12 Denial Codes
#
# This is the **only** notebook that performs cleaning, EDA, feature engineering and
# aggregation. Its outputs are the model-ready datasets used by the three model
# notebooks (`02_`, `03_`, `04_`).
#
# ### Important facts discovered during dataset inspection (Phase 1)
# * `claims_main.csv` has **120,000 rows / 25 columns** and **no duplicate `claim_id`**.
# * There is **NO reimbursement / paid-amount column** in the dataset. The only
#   monetary column is `claim_amount_usd` (the *billed* charge).
# * `denial_category` and `denial_reason_code` are populated **only for `denied` claims**
#   (33,664 rows), and `denial_labels.csv` has exactly the same 33,664 rows.
# * `denial_labels.csv` contains the real regression labels `estimated_recovery_usd`
#   and `appeal_success_probability`.
#
# ### Target decisions (agreed with the project owner)
# 1. **Model 1 (classification):** target `is_denied` -> `denied = 1`,
#    `paid / partial_pay = 0`, and `pending` rows are **excluded** (outcome undetermined).
# 2. **Model 2 (regression):** the dataset has no paid amount, so we train on the real
#    column `estimated_recovery_usd` (denied claims only) and clearly document the
#    limitation. We never fabricate a reimbursement target.
# 3. **Model 3 (multiclass):** target `denial_category` (7 classes, denied claims only).


# ## 1. Setup


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # keep plots headless-safe; notebooks still display them
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 4)
RANDOM_STATE = 42

# Locate the project root regardless of whether the notebook runs from
# the project root or from the notebooks/ folder.
BASE = Path.cwd()
while not (BASE / "data" / "raw").exists() and BASE != BASE.parent:
    BASE = BASE.parent

RAW = BASE / "data" / "raw"
PROCESSED = BASE / "data" / "processed"
REPORTS = BASE / "reports"
PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)
print("Project root :", BASE)
print("Raw data dir :", RAW)


Project root : D:\CDAC\Projects\ML_Project_Final\medical-claim-denial-predictor
Raw data dir : D:\CDAC\Projects\ML_Project_Final\medical-claim-denial-predictor\data\raw


# ## 2. Load the raw files
#
# The original files are preserved in `data/raw/` and are never overwritten.


In [2]:
claims = pd.read_csv(RAW / "claims_main.csv")
denial_labels = pd.read_csv(RAW / "denial_labels.csv")
payer_rules = pd.read_csv(RAW / "payer_rules (1).csv")
split = pd.read_csv(RAW / "train_test_split.csv")

print("claims_main     :", claims.shape)
print("denial_labels   :", denial_labels.shape)
print("payer_rules     :", payer_rules.shape)
print("train_test_split:", split.shape)


claims_main     : (120000, 25)
denial_labels   : (33664, 9)
payer_rules     : (294, 8)
train_test_split: (120000, 3)


# ## 3. Dataset shape and column inspection


In [3]:
print("Shape:", claims.shape)
print("\nColumns and dtypes:\n")
print(claims.dtypes)
claims.head()


Shape: (120000, 25)

Columns and dtypes:

claim_id                       object
claim_submission_date          object
claim_year                      int64
claim_quarter                  object
payer_type                     object
provider_specialty             object
place_of_service_code           int64
place_of_service_desc          object
cpt_code                       object
modifier                       object
primary_icd10_dx               object
primary_icd10_desc             object
secondary_icd10_dx             object
secondary_dx_count              int64
prior_auth_required              bool
prior_auth_obtained            object
prior_auth_number              object
documentation_completeness    float64
claim_amount_usd              float64
outcome                        object
denial_reason_code             object
denial_category                object
dataset_version               float64
synthetic_flag                   bool
generation_date                object
dtype: o

,claim_id,claim_submission_date,claim_year,claim_quarter,payer_type,provider_specialty,place_of_service_code,place_of_service_desc,cpt_code,modifier,primary_icd10_dx,primary_icd10_desc,secondary_icd10_dx,secondary_dx_count,prior_auth_required,prior_auth_obtained,prior_auth_number,documentation_completeness,claim_amount_usd,outcome,denial_reason_code,denial_category,dataset_version,synthetic_flag,generation_date
0,e46071e9-0fe0-42d8-ab46-dbb0c48341c4,2022-04-22,2022,Q2,Commercial_PPO,Emergency_Medicine,23,Emergency_Room,99283,NaN,G89.29,Other_chronic_pain,NaN,0,False,NaN,NaN,0.40,656.22,denied,CO-4,coding_error,2.0,True,2026-04-28
1,a641a4ce-38d4-4467-a9d3-d4970735ecc0,2022-07-24,2022,Q3,Medicare_FFS,Internal_Medicine,11,Office,99215,25,K57.30,Diverticulosis_without_perforation,NaN,0,False,NaN,NaN,0.37,332.89,denied,CO-97,duplicate,2.0,True,2026-04-28
2,2bd53273-5f36-46ad-b777-0035afd1e2e7,2022-08-24,2022,Q3,Commercial_EPO,Cardiology,2,Telehealth,93000,NaN,K57.30,Diverticulosis_without_perforation,J45.41|I48.91,2,False,NaN,NaN,0.65,185.97,denied,CO-4,bundling,2.0,True,2026-04-28
3,72df5380-e03a-4ae9-a1e0-3d3aadf13daf,2021-07-26,2021,Q3,Commercial_PPO,Physical_Therapy,11,Office,97530,59,M25.511,Pain_right_shoulder,NaN,0,False,NaN,NaN,0.85,157.25,paid,NaN,NaN,2.0,True,2026-04-28
4,9eb5d58c-3f62-40a9-971f-37855af76d48,2021-12-01,2021,Q4,Medicaid_Managed,Obstetrics_Gynecology,23,Emergency_Room,59400,NaN,J45.41,Moderate_persistent_asthma,NaN,0,True,False,NaN,0.38,8991.46,denied,CO-16,auth_missing,2.0,True,2026-04-28


In [4]:
claims.describe(include="all").T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
claim_id,120000,120000,ce3db123-aabb-4b6f-b649-14f7addde401,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
claim_submission_date,120000,1277,2023-10-15,130,NaN,NaN,NaN,NaN,NaN,NaN,NaN
claim_year,120000.0,NaN,NaN,NaN,2022.28705,1.030507,2021.0,2021.0,2022.0,2023.0,2024.0
claim_quarter,120000,4,Q2,34252,NaN,NaN,NaN,NaN,NaN,NaN,NaN
payer_type,120000,6,Commercial_PPO,30016,NaN,NaN,NaN,NaN,NaN,NaN,NaN
provider_specialty,120000,13,Emergency_Medicine,16738,NaN,NaN,NaN,NaN,NaN,NaN,NaN
place_of_service_code,120000.0,NaN,NaN,NaN,17.8877,10.274901,2.0,11.0,21.0,22.0,49.0
place_of_service_desc,120000,8,Office,36110,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cpt_code,120000,49,99214,16819,NaN,NaN,NaN,NaN,NaN,NaN,NaN
modifier,65753,6,25,24023,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# ## 4. Missing-value analysis
#
# `modifier`, `prior_auth_obtained`, `prior_auth_number`, `secondary_icd10_dx`,
# `denial_reason_code` and `denial_category` are intentionally nullable (they only
# apply to some claims).


In [5]:
missing = pd.DataFrame({
    "missing": claims.isna().sum(),
    "missing_pct": (claims.isna().mean() * 100).round(2),
}).sort_values("missing", ascending=False)
missing


,missing,missing_pct
denial_category,86336,71.95
denial_reason_code,86336,71.95
prior_auth_number,73320,61.10
prior_auth_obtained,62488,52.07
modifier,54247,45.21
secondary_icd10_dx,35928,29.94
claim_year,0,0.00
claim_submission_date,0,0.00
claim_id,0,0.00
cpt_code,0,0.00


# ## 5. Duplicate detection


In [6]:
print("Duplicate claim_id rows :", claims["claim_id"].duplicated().sum())
print("Fully duplicated rows   :", claims.duplicated().sum())


Duplicate claim_id rows : 0


Fully duplicated rows   : 0


# ## 6. Data-type correction
#
# * `claim_submission_date` -> datetime
# * `prior_auth_obtained` -> proper boolean (nullable)
# * `dataset_version` -> string label


In [7]:
claims["claim_submission_date"] = pd.to_datetime(claims["claim_submission_date"])
claims["prior_auth_obtained"] = claims["prior_auth_obtained"].map(
    {True: True, False: False, "True": True, "False": False}
)
claims["dataset_version"] = claims["dataset_version"].astype(str)
print(claims[["claim_submission_date", "prior_auth_required", "prior_auth_obtained"]].dtypes)


claim_submission_date    datetime64[ns]
prior_auth_required                bool
prior_auth_obtained              object
dtype: object


# ## 7. Outlier analysis (IQR) on numeric columns


In [8]:
def iqr_outliers(series: pd.Series) -> dict:
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return {
        "q1": round(q1, 2),
        "q3": round(q3, 2),
        "lower_fence": round(low, 2),
        "upper_fence": round(high, 2),
        "n_outliers": int(((series < low) | (series > high)).sum()),
    }

for col in ["claim_amount_usd", "documentation_completeness", "secondary_dx_count"]:
    print(col, iqr_outliers(claims[col]))


claim_amount_usd {'q1': 242.73, 'q3': 2556.86, 'lower_fence': -3228.46, 'upper_fence': 6028.04, 'n_outliers': 11619}
documentation_completeness {'q1': 0.58, 'q3': 0.88, 'lower_fence': 0.13, 'upper_fence': 1.33, 'n_outliers': 0}
secondary_dx_count {'q1': 0.0, 'q3': 2.0, 'lower_fence': -3.0, 'upper_fence': 5.0, 'n_outliers': 0}


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(x=claims["claim_amount_usd"], ax=axes[0])
axes[0].set_title("claim_amount_usd (billed charge)")
sns.boxplot(x=claims["documentation_completeness"], ax=axes[1])
axes[1].set_title("documentation_completeness")
plt.tight_layout()
plt.savefig(REPORTS / "01_outliers.png", dpi=110)
plt.show()
print("NOTE: claim_amount_usd is right-skewed. Values are kept as-is (they are valid "
      "high-cost claims); we only add a log-transformed feature for the models.")


NOTE: claim_amount_usd is right-skewed. Values are kept as-is (they are valid high-cost claims); we only add a log-transformed feature for the models.


C:\Windows\Temp\ipykernel_28540\138318605.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# ## 8. Categorical value analysis


In [10]:
for col in ["payer_type", "provider_specialty", "place_of_service_desc", "claim_quarter",
            "modifier", "cpt_code", "primary_icd10_dx"]:
    print(f"--- {col} ({claims[col].nunique()} unique) ---")
    print(claims[col].value_counts(dropna=False).head(10))
    print()


--- payer_type (6 unique) ---
payer_type
Commercial_PPO        30016
Medicare_FFS          26300
Medicare_Advantage    21758
Medicaid_Managed      17996
Commercial_HMO        15458
Commercial_EPO         8472
Name: count, dtype: int64

--- provider_specialty (13 unique) ---
provider_specialty
Emergency_Medicine    16738
Internal_Medicine     14386
Cardiology            12078
Orthopedic_Surgery    10658
Behavioral_Health      9379
Radiology              8471
Oncology               8458
Gastroenterology       7410
Primary_Care           7313
Neurology              7075
Name: count, dtype: int64

--- place_of_service_desc (8 unique) ---
place_of_service_desc
Office                      36110
Inpatient_Hospital          24027
Outpatient_Hospital         17824
Telehealth                  12087
Emergency_Room              12056
Independent_Clinic           6013
Skilled_Nursing_Facility     5996
Home                         5887
Name: count, dtype: int64

--- claim_quarter (4 unique) ---
clai

# ## 9. Numerical distribution analysis


In [11]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(claims["claim_amount_usd"], bins=50, ax=axes[0])
axes[0].set_title("claim_amount_usd")
sns.histplot(claims["documentation_completeness"], bins=30, ax=axes[1])
axes[1].set_title("documentation_completeness")
sns.countplot(x=claims["secondary_dx_count"], ax=axes[2])
axes[2].set_title("secondary_dx_count")
plt.tight_layout()
plt.savefig(REPORTS / "01_numeric_distributions.png", dpi=110)
plt.show()


C:\Windows\Temp\ipykernel_28540\250226557.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# ## 10. Correlation analysis


In [12]:
num_cols = ["claim_year", "secondary_dx_count", "documentation_completeness", "claim_amount_usd"]
corr = claims[num_cols].corr()
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Numeric correlation")
plt.tight_layout()
plt.savefig(REPORTS / "01_correlation.png", dpi=110)
plt.show()


C:\Windows\Temp\ipykernel_28540\815807944.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# ## 11. Denial vs approval distribution


In [13]:
outcome_counts = claims["outcome"].value_counts()
print(outcome_counts)
print("\nOverall denial rate (denied / all claims):",
      round((claims["outcome"] == "denied").mean(), 4))

plt.figure(figsize=(7, 4))
sns.barplot(x=outcome_counts.index, y=outcome_counts.values)
plt.title("Claim outcome distribution")
plt.ylabel("Number of claims")
plt.tight_layout()
plt.savefig(REPORTS / "01_outcome_distribution.png", dpi=110)
plt.show()


outcome
paid           62379
denied         33664
partial_pay    15469
pending         8488
Name: count, dtype: int64

Overall denial rate (denied / all claims): 0.2805


C:\Windows\Temp\ipykernel_28540\2675586578.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# ## 12. Payer-wise denial analysis


In [14]:
payer_stats = (
    claims.groupby("payer_type")
    .agg(
        claims=("claim_id", "count"),
        denied=("outcome", lambda s: (s == "denied").sum()),
        avg_claim_amount=("claim_amount_usd", "mean"),
    )
    .reset_index()
)
payer_stats["denial_rate"] = (payer_stats["denied"] / payer_stats["claims"]).round(4)
payer_stats = payer_stats.sort_values("denial_rate", ascending=False)
payer_stats


,payer_type,claims,denied,avg_claim_amount,denial_rate
3,Medicaid_Managed,17996,5130,1710.245059,0.2851
4,Medicare_Advantage,21758,6148,2104.617416,0.2826
5,Medicare_FFS,26300,7378,2029.213721,0.2805
1,Commercial_HMO,15458,4324,2407.832204,0.2797
2,Commercial_PPO,30016,8338,2625.383882,0.2778
0,Commercial_EPO,8472,2346,2463.150694,0.2769


In [15]:
plt.figure(figsize=(9, 4))
sns.barplot(data=payer_stats, x="payer_type", y="denial_rate")
plt.xticks(rotation=30, ha="right")
plt.title("Denial rate by payer type")
plt.tight_layout()
plt.savefig(REPORTS / "01_payer_denial_rate.png", dpi=110)
plt.show()


C:\Windows\Temp\ipykernel_28540\783705269.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# ## 13. Denial-code analysis


In [16]:
denied = claims[claims["outcome"] == "denied"].copy()
code_counts = denied["denial_reason_code"].value_counts()
print(code_counts)

plt.figure(figsize=(10, 4))
sns.barplot(x=code_counts.index, y=code_counts.values)
plt.xticks(rotation=45, ha="right")
plt.title("Most common X12 denial reason codes")
plt.tight_layout()
plt.savefig(REPORTS / "01_denial_codes.png", dpi=110)
plt.show()


denial_reason_code
CO-11     4236
CO-16     3928
CO-109    3654
CO-97     3214
CO-4      2354
CO-119    2354
CO-50     2341
CO-29     2315
PR-204    2065
N-130     2043
PR-27     1429
CO-22     1331
OA-18      900
CO-18      889
OA-23      611
Name: count, dtype: int64


C:\Windows\Temp\ipykernel_28540\4095153802.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
cat_counts = denied["denial_category"].value_counts()
plt.figure(figsize=(8, 4))
sns.barplot(x=cat_counts.index, y=cat_counts.values)
plt.xticks(rotation=30, ha="right")
plt.title("Denial category distribution")
plt.tight_layout()
plt.savefig(REPORTS / "01_denial_categories.png", dpi=110)
plt.show()


C:\Windows\Temp\ipykernel_28540\323804016.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# ## 14. Claim amount & reimbursement analysis
#
# **Limitation:** the dataset records only the *billed* amount (`claim_amount_usd`).
# There is no paid/reimbursement amount for `paid` or `partial_pay` claims, so a true
# reimbursement regression target does not exist. The only real monetary outcome
# labels are `estimated_recovery_usd` (expected recovery if a denied claim is appealed)
# and `appeal_success_probability`, both in `denial_labels.csv`.


In [18]:
print("Billed amount by outcome:")
print(claims.groupby("outcome")["claim_amount_usd"].describe()[["count", "mean", "50%", "max"]])
print("\nRecovery / appeal labels (denied claims only):")
print(denial_labels[["appeal_success_probability", "estimated_recovery_usd"]].describe())


Billed amount by outcome:
               count         mean      50%       max
outcome                                             
denied       33664.0  2202.125953  431.160  37897.59
paid         62379.0  2229.197564  432.990  37796.11
partial_pay  15469.0  2255.852252  439.950  37632.91
pending       8488.0  2208.588689  434.755  37918.58

Recovery / appeal labels (denied claims only):
       appeal_success_probability  estimated_recovery_usd
count                33664.000000            33664.000000
mean                     0.430868              965.580381
std                      0.240449             2125.715030
min                      0.000000                0.000000
25%                      0.310000               78.390000
50%                      0.450000              178.065000
75%                      0.590000              946.085000
max                      0.840000            30265.170000


# ## 15. Merge auxiliary files
#
# We join the split assignment, the denial labels and the external payer rule table.


In [19]:
df = claims.merge(split[["claim_id", "split"]], on="claim_id", how="left")
df = df.merge(
    denial_labels[["claim_id", "appealable", "appeal_success_probability",
                   "recovery_action", "estimated_recovery_usd"]],
    on="claim_id", how="left",
)
df = df.merge(
    payer_rules[["payer_type", "cpt_code", "historical_denial_rate",
                 "auth_lead_time_days", "avg_payment_turnaround_days",
                 "timely_filing_limit_days"]],
    on=["payer_type", "cpt_code"], how="left",
)
print(df.shape)
print("split values:", df["split"].value_counts().to_dict())


(120000, 34)
split values: {'train': 84051, 'val': 18018, 'test': 17931}


# ## 16. Feature engineering
#
# All engineered features are derived from input columns only (no target leakage).


In [20]:
# --- date features ---
df["claim_month"] = df["claim_submission_date"].dt.month
df["claim_dayofweek"] = df["claim_submission_date"].dt.dayofweek

# --- authorisation features ---
# prior_auth_obtained is null when authorisation is not required: collapse into one
# clean categorical status.
def auth_status(row):
    if not row["prior_auth_required"]:
        return "not_required"
    if row["prior_auth_obtained"] is True or row["prior_auth_obtained"] == True:
        return "obtained"
    if row["prior_auth_obtained"] is False or row["prior_auth_obtained"] == False:
        return "not_obtained"
    return "unknown"

df["auth_status"] = df.apply(auth_status, axis=1)
df["pa_gap"] = (
    (df["prior_auth_required"] == True) & (df["auth_status"] != "obtained")
).astype(int)
df["prior_auth_required_int"] = df["prior_auth_required"].astype(int)

# --- documentation features ---
df["doc_low"] = (df["documentation_completeness"] < 0.6).astype(int)

# --- modifier / secondary diagnosis features ---
df["has_modifier"] = df["modifier"].notna().astype(int)
df["modifier"] = df["modifier"].fillna("none")
df["has_secondary_dx"] = (df["secondary_dx_count"] > 0).astype(int)

# --- amount features ---
df["log_claim_amount"] = np.log1p(df["claim_amount_usd"])

print(df[["claim_month", "claim_dayofweek", "auth_status", "pa_gap",
          "has_modifier", "has_secondary_dx", "log_claim_amount"]].head())


   claim_month  claim_dayofweek   auth_status  pa_gap  has_modifier  has_secondary_dx  log_claim_amount
0            4                4  not_required       0             0                 0          6.488019
1            7                6  not_required       0             1                 0          5.810812
2            8                2  not_required       0             0                 1          5.230948
3            7                0  not_required       0             1                 0          5.064176
4           12                2  not_obtained       1             0                 0          9.104142


# ## 17. Historical payer intelligence (leakage-safe)
#
# Historical denial rates are computed **from the training split only** and then mapped
# onto every row. This mirrors real deployment (you only know history for past claims)
# and prevents the target from leaking into the features.
#
# The external `payer_rules.historical_denial_rate` is also merged as
# `payer_rule_denial_rate` (it is a provided reference table, not derived from labels).


In [21]:
train_mask = df["split"] == "train"
train_df = df[train_mask].copy()
train_df["is_denied_int"] = (train_df["outcome"] == "denied").astype(int)
overall_train_denial_rate = float(train_df["is_denied_int"].mean())
print("Overall train denial rate:", round(overall_train_denial_rate, 4))

# Full-train aggregates. These are used for validation/test rows and, later, at
# inference time (a brand-new claim is "in the future", so train history is all we know).
payer_hist = (
    train_df.groupby("payer_type")
    .agg(payer_hist_claims=("claim_id", "count"),
         payer_denial_rate_full=("is_denied_int", "mean"),
         payer_avg_amount_hist=("claim_amount_usd", "mean"))
    .reset_index()
)
payer_cpt_hist = (
    train_df.groupby(["payer_type", "cpt_code"])
    .agg(payer_cpt_hist_claims=("claim_id", "count"),
         payer_cpt_denial_rate_full=("is_denied_int", "mean"))
    .reset_index()
)

# For the TRAIN rows we use 5-fold out-of-fold encoding so that a claim's own label
# never contributes to its own feature. This removes target leakage.
rng = np.random.RandomState(RANDOM_STATE)
train_df["fold"] = rng.randint(0, 5, size=len(train_df))


def oof_denial_rate(frame, keys, n_folds=5):
    """Return an out-of-fold mean of is_denied_int for the given grouping keys."""
    out = pd.Series(np.nan, index=frame.index, dtype=float)
    for f in range(n_folds):
        val_idx = frame.index[frame["fold"] == f]
        stats = (frame[frame["fold"] != f]
                 .groupby(keys)["is_denied_int"].mean()
                 .rename("rate").reset_index())
        tmp = frame.loc[val_idx, keys].copy()
        tmp["_i"] = tmp.index
        tmp = tmp.merge(stats, on=keys, how="left").set_index("_i")
        out.loc[tmp.index] = tmp["rate"].values
    return out


train_df["oof_payer"] = oof_denial_rate(train_df, ["payer_type"])
train_df["oof_payer_cpt"] = oof_denial_rate(train_df, ["payer_type", "cpt_code"])

df = df.merge(payer_hist, on="payer_type", how="left")
df = df.merge(payer_cpt_hist, on=["payer_type", "cpt_code"], how="left")
df = df.merge(train_df[["claim_id", "oof_payer", "oof_payer_cpt"]], on="claim_id", how="left")

# train rows -> out-of-fold value; other rows -> full-train value
df["payer_denial_rate_hist"] = df["oof_payer"].where(df["oof_payer"].notna(), df["payer_denial_rate_full"])
df["payer_cpt_denial_rate_hist"] = df["oof_payer_cpt"].where(df["oof_payer_cpt"].notna(), df["payer_cpt_denial_rate_full"])

# fall back to the overall rate when a payer or payer x cpt combination is unseen
df["payer_denial_rate_hist"] = df["payer_denial_rate_hist"].fillna(overall_train_denial_rate)
df["payer_cpt_denial_rate_hist"] = df["payer_cpt_denial_rate_hist"].fillna(df["payer_denial_rate_hist"])
df["payer_hist_claims"] = df["payer_hist_claims"].fillna(0)
df["payer_cpt_hist_claims"] = df["payer_cpt_hist_claims"].fillna(0)
df["payer_avg_amount_hist"] = df["payer_avg_amount_hist"].fillna(df["claim_amount_usd"].mean())

# external payer rule table (provided reference data, not derived from labels)
df["payer_rule_denial_rate"] = df["historical_denial_rate"]
df = df.rename(columns={"auth_lead_time_days": "payer_rule_auth_lead_time_days",
                        "avg_payment_turnaround_days": "payer_rule_turnaround_days",
                        "timely_filing_limit_days": "payer_rule_timely_filing_days"})

print(payer_hist)
print(payer_cpt_hist.head())


Overall train denial rate: 0.2807


           payer_type  payer_hist_claims  payer_denial_rate_full  payer_avg_amount_hist
0      Commercial_EPO               5792                0.276416            2511.128700
1      Commercial_HMO              10762                0.278852            2391.155879
2      Commercial_PPO              21077                0.277269            2611.729007
3    Medicaid_Managed              12649                0.286900            1699.210522
4  Medicare_Advantage              15323                0.283756            2127.455472
5        Medicare_FFS              18448                0.280464            2023.229851
       payer_type cpt_code  payer_cpt_hist_claims  payer_cpt_denial_rate_full
0  Commercial_EPO    20610                    116                    0.267241
1  Commercial_EPO    27447                    107                    0.271028
2  Commercial_EPO    29827                    113                    0.238938
3  Commercial_EPO    29881                    114                    0.3

# ## 18. Define targets


In [22]:
# Model 1 binary target (pending excluded later at training time)
df["is_denied"] = np.where(df["outcome"] == "denied", 1,
                    np.where(df["outcome"].isin(["paid", "partial_pay"]), 0, np.nan))

# Model 3 target already exists as denial_category (denied rows only)

print(df.groupby("outcome")["is_denied"].apply(lambda s: s.notna().sum()))


outcome
denied         33664
paid           62379
partial_pay    15469
pending            0
Name: is_denied, dtype: int64


# ## 19. Feature lists (shared by all model notebooks and the FastAPI backend)


In [23]:
CATEGORICAL_FEATURES = [
    "payer_type",
    "provider_specialty",
    "place_of_service_code",
    "cpt_code",
    "modifier",
    "primary_icd10_dx",
    "claim_quarter",
    "auth_status",
]

NUMERIC_FEATURES = [
    "claim_year",
    "claim_month",
    "claim_dayofweek",
    "secondary_dx_count",
    "documentation_completeness",
    "claim_amount_usd",
    "log_claim_amount",
    "has_secondary_dx",
    "has_modifier",
    "prior_auth_required_int",
    "pa_gap",
    "doc_low",
    "payer_denial_rate_hist",
    "payer_cpt_denial_rate_hist",
    "payer_hist_claims",
    "payer_cpt_hist_claims",
    "payer_avg_amount_hist",
    "payer_rule_denial_rate",
    "payer_rule_auth_lead_time_days",
    "payer_rule_turnaround_days",
    "payer_rule_timely_filing_days",
]
FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES
print(len(CATEGORICAL_FEATURES), "categorical +", len(NUMERIC_FEATURES), "numeric =", len(FEATURES))


8 categorical + 21 numeric = 29


# ## 19b. Feature selection
#
# The full engineered set has 29 features. That is more than a billing clerk can enter
# for a single claim, and most of them carry almost no signal. We therefore run a
# **multi-model feature-selection** step and keep only a small, directly enterable set.
#
# Method:
# 1. Train three quick XGBoost models (denial, recovery, reason) on the **train split**
#    using the full feature set.
# 2. Aggregate the XGBoost **gain** importance of the one-hot columns back to their
#    original feature, per model.
# 3. Combine the three rankings (mean gain share), restricted to the **directly
#    enterable** claim attributes (a user can type these in a form without any hidden
#    engineering step).
# 4. Keep the **top 5** and retrain every model on only those 5 features.
#
# Restricting the candidates to enterable attributes is what makes a genuinely usable
# "Top-5 feature" form possible.


In [24]:
from sklearn.compose import ColumnTransformer  # noqa: E402
from sklearn.pipeline import Pipeline  # noqa: E402
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder  # noqa: E402
from sklearn.impute import SimpleImputer  # noqa: E402
from xgboost import XGBClassifier, XGBRegressor  # noqa: E402

# A user can provide these directly; everything else is derived/engineered.
INPUTTABLE = [
    "payer_type", "provider_specialty", "place_of_service_code", "cpt_code",
    "modifier", "primary_icd10_dx", "auth_status", "secondary_dx_count",
    "documentation_completeness", "claim_amount_usd",
]


def _quick_preprocessor():
    num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),
                         ("scaler", StandardScaler())])
    cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                         ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    return ColumnTransformer([("num", num_pipe, NUMERIC_FEATURES),
                              ("cat", cat_pipe, CATEGORICAL_FEATURES)])


def _gain_importance(pipeline, label):
    """Aggregate one-hot gain importance back to the original feature, as a share (%)."""
    names = pipeline.named_steps["prep"].get_feature_names_out()
    est = pipeline.named_steps.get("clf") or pipeline.named_steps.get("reg")
    gains = pd.Series(est.feature_importances_, index=names)
    share = {}
    for feat in FEATURES:
        cols = [c for c in names if c == f"num__{feat}" or c == f"cat__{feat}"
                or c.startswith(f"cat__{feat}_")]
        share[feat] = float(gains[cols].sum()) if cols else 0.0
    s = pd.Series(share).loc[INPUTTABLE]
    s = s / s.sum() * 100.0
    return s.round(3).rename(label)


# --- quick denial model ---
d_rows = df[df["is_denied"].notna()]
d_tr = d_rows["split"] == "train"
denial_quick = Pipeline([
    ("prep", _quick_preprocessor()),
    ("clf", XGBClassifier(objective="binary:logistic", n_estimators=200, max_depth=3,
                          learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                          tree_method="hist", random_state=RANDOM_STATE)),
])
denial_quick.fit(d_rows[d_tr][FEATURES], d_rows[d_tr]["is_denied"].astype(int))

# --- quick recovery model ---
r_rows = df[(df["outcome"] == "denied") & (df["estimated_recovery_usd"].notna())]
r_tr = r_rows["split"] == "train"
recovery_quick = Pipeline([
    ("prep", _quick_preprocessor()),
    ("reg", XGBRegressor(objective="reg:squarederror", n_estimators=300, max_depth=4,
                         learning_rate=0.05, subsample=0.9, colsample_bytree=0.7,
                         tree_method="hist", random_state=RANDOM_STATE)),
])
recovery_quick.fit(r_rows[r_tr][FEATURES], r_rows[r_tr]["estimated_recovery_usd"])

# --- quick reason model ---
q_rows = df[df["denial_category"].notna()].copy()
q_tr = q_rows["split"] == "train"
_le = LabelEncoder()
q_rows["y"] = _le.fit_transform(q_rows["denial_category"])
reason_quick = Pipeline([
    ("prep", _quick_preprocessor()),
    ("clf", XGBClassifier(objective="multi:softprob", n_estimators=300, max_depth=8,
                          learning_rate=0.2, subsample=1.0, colsample_bytree=0.8,
                          tree_method="hist", random_state=RANDOM_STATE)),
])
reason_quick.fit(q_rows[q_tr][FEATURES], q_rows[q_tr]["y"])


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](7,)","[0,1,2,...,4,5,6]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](29,)","['payer_type','provider_specialty','place_of_service_code',..., 'payer_rule_auth_lead_time_days','payer_rule_turnaround_days', 'payer_rule_timely_filing_days']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,29
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedco

In [25]:
importance = pd.DataFrame({
    "denial_pct": _gain_importance(denial_quick, "denial_pct"),
    "recovery_pct": _gain_importance(recovery_quick, "recovery_pct"),
    "reason_pct": _gain_importance(reason_quick, "reason_pct"),
})
importance["combined_pct"] = importance.mean(axis=1).round(3)
importance = importance.sort_values("combined_pct", ascending=False)
print("Multi-model gain importance over directly enterable features:")
print(importance)

plt.figure(figsize=(8, 5))
sns.barplot(data=importance.reset_index().rename(columns={"index": "feature"}),
            y="feature", x="combined_pct")
plt.title("Feature selection - combined importance (inputtable features)")
plt.xlabel("Mean gain importance (%)")
plt.tight_layout(); plt.savefig(REPORTS / "01_feature_selection.png", dpi=110); plt.show()


Multi-model gain importance over directly enterable features:
                            denial_pct  recovery_pct  reason_pct  combined_pct
documentation_completeness      76.233         2.011       1.578        26.607
primary_icd10_dx                 7.387        28.919      25.759        20.688
cpt_code                         5.852        15.983      36.791        19.542
modifier                         1.563        12.934       5.598         6.698
place_of_service_code            1.468        11.471       6.275         6.405
auth_status                      3.722         5.275       6.571         5.189
provider_specialty               2.907         1.826      10.610         5.114
claim_amount_usd                 0.345        13.640       0.743         4.909
payer_type                       0.250         6.993       5.360         4.201
secondary_dx_count               0.272         0.947       0.715         0.645


C:\Windows\Temp\ipykernel_28540\1333728931.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(REPORTS / "01_feature_selection.png", dpi=110); plt.show()


# **Final selection.** The data-driven ranking above is dominated by `cpt_code`,
# `primary_icd10_dx` and `documentation_completeness`. The last two slots were validated
# by retraining all three models on the candidate sets: including `claim_amount_usd`
# (the recovery model's strongest input) and `auth_status` (the denial/reason models'
# strongest behavioural signal) gave the best combined accuracy, so they are kept in
# place of `modifier` / `place_of_service_code`.


In [26]:
SELECTED_FEATURES = [
    "documentation_completeness",   # how complete the clinical documentation is
    "cpt_code",                     # procedure performed
    "primary_icd10_dx",             # primary diagnosis
    "claim_amount_usd",             # billed charge
    "auth_status",                  # prior-authorisation state
]
SELECTED_CATEGORICAL = [f for f in SELECTED_FEATURES if f in CATEGORICAL_FEATURES]
SELECTED_NUMERIC = [f for f in SELECTED_FEATURES if f in NUMERIC_FEATURES]
print("Selected features :", SELECTED_FEATURES)
print(" categorical:", SELECTED_CATEGORICAL)
print(" numeric    :", SELECTED_NUMERIC)
print(f"Feature count reduced from {len(FEATURES)} to {len(SELECTED_FEATURES)}")


Selected features : ['documentation_completeness', 'cpt_code', 'primary_icd10_dx', 'claim_amount_usd', 'auth_status']
 categorical: ['cpt_code', 'primary_icd10_dx', 'auth_status']
 numeric    : ['documentation_completeness', 'claim_amount_usd']
Feature count reduced from 29 to 5


# ## 20. Build and save the model-ready datasets


In [27]:
keep_cols = (["claim_id", "split", "outcome", "is_denied", "denial_category",
              "denial_reason_code", "estimated_recovery_usd", "appeal_success_probability",
              "appealable", "recovery_action", "claim_submission_date"]
             + FEATURES)
model_ready = df[keep_cols].copy()
model_ready.to_csv(PROCESSED / "claims_model_ready.csv", index=False)
print("claims_model_ready.csv:", model_ready.shape)

# cleaned full data (all 120k, extra descriptive columns kept for the dashboard)
df.to_csv(PROCESSED / "claims_clean.csv", index=False)
print("claims_clean.csv:", df.shape)


claims_model_ready.csv: (120000, 40)


claims_clean.csv: (120000, 54)


# ## 21. Aggregations for payer intelligence & dashboard


In [28]:
# payer intelligence (aggregated over the whole dataset, used for display only)
payer_agg = (
    df.groupby("payer_type")
    .agg(claims=("claim_id", "count"),
         denied=("outcome", lambda s: (s == "denied").sum()),
         paid=("outcome", lambda s: (s == "paid").sum()),
         partial_pay=("outcome", lambda s: (s == "partial_pay").sum()),
         pending=("outcome", lambda s: (s == "pending").sum()),
         avg_claim_amount=("claim_amount_usd", "mean"),
         avg_recovery_usd=("estimated_recovery_usd", "mean"))
    .reset_index()
)
payer_agg["denial_rate"] = (payer_agg["denied"] / payer_agg["claims"]).round(4)
payer_agg["approval_rate"] = ((payer_agg["paid"] + payer_agg["partial_pay"]) / payer_agg["claims"]).round(4)
payer_agg = payer_agg.round(2)

# most common denial code + category per payer
top_code = (denied.groupby(["payer_type", "denial_reason_code"]).size()
            .reset_index(name="n").sort_values("n", ascending=False)
            .drop_duplicates("payer_type")[["payer_type", "denial_reason_code"]])
top_cat = (denied.groupby(["payer_type", "denial_category"]).size()
           .reset_index(name="n").sort_values("n", ascending=False)
           .drop_duplicates("payer_type")[["payer_type", "denial_category"]])
payer_agg = payer_agg.merge(top_code, on="payer_type", how="left")
payer_agg = payer_agg.merge(top_cat, on="payer_type", how="left")
payer_agg.to_csv(PROCESSED / "payer_aggregates.csv", index=False)
print(payer_agg)


           payer_type  claims  denied   paid  partial_pay  pending  avg_claim_amount  avg_recovery_usd  denial_rate  approval_rate denial_reason_code    denial_category
0      Commercial_EPO    8472    2346   4405         1122      599           2463.15           1125.39         0.28           0.65              CO-16  medical_necessity
1      Commercial_HMO   15458    4324   8030         2017     1087           2407.83           1172.87         0.28           0.65              CO-11  medical_necessity
2      Commercial_PPO   30016    8338  15651         3876     2151           2625.38           1261.54         0.28           0.65              CO-11  medical_necessity
3    Medicaid_Managed   17996    5130   9300         2260     1306           1710.25            568.91         0.29           0.64              CO-11  medical_necessity
4  Medicare_Advantage   21758    6148  11335         2729     1546           2104.62            843.54         0.28           0.65              CO-11  medi

In [29]:
# denial-code aggregates (use the merged frame so the recovery/appeal labels are available)
denied_full = df[df["outcome"] == "denied"]
code_agg = (
    denied_full.groupby(["denial_reason_code", "denial_category"])
    .agg(claims=("claim_id", "count"),
         avg_claim_amount=("claim_amount_usd", "mean"),
         avg_recovery_usd=("estimated_recovery_usd", "mean"),
         appealable_rate=("appealable", "mean"))
    .reset_index()
    .sort_values("claims", ascending=False)
    .round(2)
)
code_agg.to_csv(PROCESSED / "denial_code_aggregates.csv", index=False)
print(code_agg)


   denial_reason_code    denial_category  claims  avg_claim_amount  avg_recovery_usd appealable_rate
3               CO-11  medical_necessity    2419           2209.26           1058.10             1.0
4              CO-119  medical_necessity    2354           2094.13            992.16             1.0
12              CO-50  medical_necessity    2341           2177.04           1041.96             1.0
1              CO-109  medical_necessity    2330           2285.42           1093.33             1.0
9               CO-29      timely_filing    2315           2083.60              0.00             0.0
19             PR-204       auth_missing    2065           2044.39            830.46             1.0
16              N-130       auth_missing    2043           2373.44            973.63             1.0
5               CO-16       auth_missing    2032           2313.55            950.38             1.0
6               CO-16       coding_error    1896           2074.26           1551.69       

# ## 22. Save reusable lookup tables (needed by the FastAPI backend at inference)


In [30]:
payer_hist.to_csv(PROCESSED / "lookup_payer.csv", index=False)
payer_cpt_hist.to_csv(PROCESSED / "lookup_payer_cpt.csv", index=False)
payer_rules.to_csv(PROCESSED / "lookup_payer_rules.csv", index=False)

# quantiles used by the API to flag "high billed amount" style risk factors
quantiles = {
    "claim_amount_q25": round(float(df["claim_amount_usd"].quantile(0.25)), 2),
    "claim_amount_q50": round(float(df["claim_amount_usd"].quantile(0.50)), 2),
    "claim_amount_q75": round(float(df["claim_amount_usd"].quantile(0.75)), 2),
    "documentation_completeness_q25": round(float(df["documentation_completeness"].quantile(0.25)), 4),
    "documentation_completeness_q75": round(float(df["documentation_completeness"].quantile(0.75)), 4),
}

with open(PROCESSED / "feature_columns.json", "w", encoding="utf-8") as fh:
    json.dump({"categorical": CATEGORICAL_FEATURES,
               "numeric": NUMERIC_FEATURES,
               "features": FEATURES,
               "selected_features": SELECTED_FEATURES,
               "selected_categorical": SELECTED_CATEGORICAL,
               "selected_numeric": SELECTED_NUMERIC,
               "feature_selection": {
                   "method": "multi-model XGBoost gain importance over enterable features",
                   "ranking": {k: v for k, v in
                               importance["combined_pct"].to_dict().items()},
                   "n_features_full": len(FEATURES),
                   "n_features_selected": len(SELECTED_FEATURES),
               },
               "overall_train_denial_rate": float(overall_train_denial_rate),
               "quantiles": quantiles}, fh, indent=2)
print("quantiles:", quantiles)
print("selected_features written to feature_columns.json")


quantiles: {'claim_amount_q25': 242.73, 'claim_amount_q50': 433.27, 'claim_amount_q75': 2556.86, 'documentation_completeness_q25': 0.58, 'documentation_completeness_q75': 0.88}
selected_features written to feature_columns.json


# ## 23. Dashboard summary statistics (JSON for the Streamlit dashboard)


In [31]:
dashboard = {
    "total_claims": int(len(df)),
    "denied_claims": int((df["outcome"] == "denied").sum()),
    "paid_claims": int((df["outcome"] == "paid").sum()),
    "partial_pay_claims": int((df["outcome"] == "partial_pay").sum()),
    "pending_claims": int((df["outcome"] == "pending").sum()),
    "unique_payers": int(df["payer_type"].nunique()),
    "overall_denial_rate": round(float((df["outcome"] == "denied").mean()), 4),
    "overall_approval_rate": round(float(df["outcome"].isin(["paid", "partial_pay"]).mean()), 4),
    "avg_claim_amount": round(float(df["claim_amount_usd"].mean()), 2),
    "avg_recovery_usd": round(float(df["estimated_recovery_usd"].mean()), 2),
    "outcome_counts": {k: int(v) for k, v in df["outcome"].value_counts().items()},
    "denial_category_counts": {k: int(v) for k, v in denied["denial_category"].value_counts().items()},
    "denial_code_counts": {k: int(v) for k, v in denied["denial_reason_code"].value_counts().items()},
    "payer_denial_rates": {r["payer_type"]: float(r["denial_rate"])
                           for _, r in payer_agg.iterrows()},
    "monthly_claims": {int(k): int(v) for k, v in
                       df.groupby("claim_month")["claim_id"].count().items()},
    "monthly_denials": {int(k): int(v) for k, v in
                        denied_full.groupby("claim_month")["claim_id"].count().items()},
    "claim_amount_by_outcome": {k: round(float(v), 2) for k, v in
                                df.groupby("outcome")["claim_amount_usd"].mean().items()},
}
with open(PROCESSED / "dashboard_stats.json", "w", encoding="utf-8") as fh:
    json.dump(dashboard, fh, indent=2)
dashboard


{'total_claims': 120000,
 'denied_claims': 33664,
 'paid_claims': 62379,
 'partial_pay_claims': 15469,
 'pending_claims': 8488,
 'unique_payers': 6,
 'overall_denial_rate': 0.2805,
 'overall_approval_rate': 0.6487,
 'avg_claim_amount': 2223.58,
 'avg_recovery_usd': 965.58,
 'outcome_counts': {'paid': 62379,
  'denied': 33664,
  'partial_pay': 15469,
  'pending': 8488},
 'denial_category_counts': {'medical_necessity': 9444,
  'coding_error': 7344,
  'auth_missing': 6140,
  'eligibility': 4084,
  'duplicate': 2645,
  'timely_filing': 2315,
  'bundling': 1692},
 'denial_code_counts': {'CO-11': 4236,
  'CO-16': 3928,
  'CO-109': 3654,
  'CO-97': 3214,
  'CO-4': 2354,
  'CO-119': 2354,
  'CO-50': 2341,
  'CO-29': 2315,
  'PR-204': 2065,
  'N-130': 2043,
  'PR-27': 1429,
  'CO-22': 1331,
  'OA-18': 900,
  'CO-18': 889,
  'OA-23': 611},
 'payer_denial_rates': {'Commercial_EPO': 0.28,
  'Commercial_HMO': 0.28,
  'Commercial_PPO': 0.28,
  'Medicaid_Managed': 0.29,
  'Medicare_Advantage': 0.28,


# ## 24. Summary of saved artefacts


In [32]:
print("Saved to", PROCESSED)
for p in sorted(PROCESSED.glob("*")):
    print(f"  {p.name:<32} {p.stat().st_size/1024:>10.1f} KB")


Saved to D:\CDAC\Projects\ML_Project_Final\medical-claim-denial-predictor\data\processed
  claims_clean.csv                    50833.1 KB
  claims_model_ready.csv              33025.4 KB
  dashboard_stats.json                    1.7 KB
  denial_code_aggregates.csv              1.0 KB
  feature_columns.json                    2.7 KB
  lookup_payer.csv                        0.4 KB
  lookup_payer_cpt.csv                   13.1 KB
  lookup_payer_rules.csv                 13.6 KB
  payer_aggregates.csv                    0.7 KB


In [33]:
print("=== PHASE 2 COMPLETE ===")
print("Model-ready rows:", len(model_ready))
print("Features         :", len(FEATURES))
print("Model 1 target   : is_denied (pending excluded at train time)")
print("Model 2 target   : estimated_recovery_usd")
print("Model 3 target   : denial_category")


=== PHASE 2 COMPLETE ===
Model-ready rows: 120000
Features         : 29
Model 1 target   : is_denied (pending excluded at train time)
Model 2 target   : estimated_recovery_usd
Model 3 target   : denial_category
